In [ ]:
from langchain_community.document_loaders import PyPDFLoader

from pathlib import Path

# Load all PDFs in the NC directory
pdf_dir = Path("../data/raw/NC/2026/formulary/")
pdf_files = list(pdf_dir.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f"  - {pdf.name}")

# Load all PDFs
all_docs = []
for pdf_path in pdf_files:
    print(f"\nLoading {pdf_path.name}...")
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    all_docs.extend(docs)
    print(f"  Loaded {len(docs)} pages")

print(f"\n✅ Total pages loaded: {len(all_docs)}")


In [ ]:
# load the documents
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    "../data/raw/NC/2026/formulary/",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)
all_docs = loader.load()
print(f"Loaded {len(all_docs)} total pages")


In [ ]:
import pprint
pprint.pprint(all_docs[0].metadata)
pprint.pprint(all_docs[0].page_content[:500])  # Print first 500 characters

pprint.pprint(all_docs[1].metadata)
pprint.pprint(all_docs[1].page_content[:500])  

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50, 
    length_function=len, 
    is_separator_regex=False,
)

In [ ]:
# Chunk all documents
chunks = text_splitter.split_documents(all_docs)
print(f"Total chunks: {len(chunks)}")




In [ ]:
### Create Embeddings & Vector Store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Rebuild vector store with ALL documents
print("Creating embeddings and FAISS index...")
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"✅ Vector store created with {len(chunks)} chunks from all PDFs")
print(vectorstore)



In [ ]:
# Test retrieval with a sample question
query = "What is the deductible for the Oscar 4T plan?"

# Retrieve top 3 most relevant chunks
relevant_docs = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
print(f"Found {len(relevant_docs)} relevant chunks:\n")

for i, doc in enumerate(relevant_docs, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:200] + "...")  # Show first 200 chars
    print(f"Metadata: {doc.metadata}\n")

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from project root
load_dotenv("../.env")

# Check if API key is loaded
api_key = os.getenv("GEMINI_API_KEY")
if api_key:
    print("✅ API key loaded")
else:
    print("❌ API key not found. Make sure .env file has GEMINI_API_KEY")


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    max_tokens=1000
)

# Create a simple prompt template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant answering questions about health insurance plans. Use only the provided context to answer."),
    ("user", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")
])

print("✅ LLM and prompt ready!")



In [ ]:
def ask_question(question):
    # 1. Retrieve relevant chunks
    docs = vectorstore.similarity_search(question, k=5)
    
    # 2. Combine chunks into context
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # 3. Create the prompt
    messages = prompt_template.format_messages(
        context=context,
        question=question
    )
    
    # 4. Get answer from LLM
    response = llm.invoke(messages)
    
    return response.content

# Test it
answer = ask_question("What pharmacy benefits are covered?")
print(answer)


In [ ]:
# Try with 5 chunks instead of 3
answer = ask_question("What is the deductible for the Oscar 4T plan?")
print(f"\nAnswer: {answer}")